# Aufgabe 4b: Lineare Single-Cell-SVM

Dieses Notebook enthält den technischen Pipeline-Test aus Planpunkt 2 und die lineare Single-Cell-SVM aus Planpunkt 3. Der kleine deterministische Test läuft zunächst auf `gated_NK`; der finale Hauptvergleich wird später mit `gated_alive` durchgeführt.

## 1. Konfiguration

`gated_NK` dient hier ausdrücklich nur als schneller technischer Test. Der finale Hauptvergleich wird später mit `gated_alive` durchgeführt.

In [1]:
from pathlib import Path
import csv
import os
import warnings

import flowkit as fk
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    auc,
    balanced_accuracy_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

RUN_MODE = os.environ.get("TASK4_RUN_MODE", "smoke")
RUN_TRAINING = os.environ.get("TASK4_RUN_TRAINING", "1") == "1"
if RUN_MODE == "smoke":
    GATE = "gated_NK"
    ANALYSIS_SPLIT_IDS = [0, 1]
    TRAIN_CELLS_PER_DONOR = 2_000
elif RUN_MODE == "full":
    GATE = "gated_alive"
    ANALYSIS_SPLIT_IDS = list(range(100))
    TRAIN_CELLS_PER_DONOR = 10_000
else:
    raise ValueError(f"Unbekannter RUN_MODE: {RUN_MODE}")
SPLIT_LIMIT = int(os.environ.get("TASK4_SPLIT_LIMIT", "0"))
if SPLIT_LIMIT > 0:
    ANALYSIS_SPLIT_IDS = ANALYSIS_SPLIT_IDS[:SPLIT_LIMIT]
FORCE_SPLIT_IDS = {
    int(value)
    for value in os.environ.get("TASK4_FORCE_SPLITS", "").split(",")
    if value.strip()
}
GATE_SUFFIXES = {"gated_NK": "_NK", "gated_alive": "_alive"}
EXPECTED_EVENT_COUNTS = {"gated_NK": 261_593, "gated_alive": 3_438_750}
if GATE not in GATE_SUFFIXES:
    raise ValueError(f"Unbekannter Gate: {GATE}")
GATE_SUFFIX = GATE_SUFFIXES[GATE]
COFACTOR = 5.0
TECHNICAL_SPLIT_IDS = [0, 1]
TECHNICAL_INNER_FOLD = 0
TECHNICAL_CELLS_PER_DONOR = 2_000
SAMPLING_SEED_OFFSET = 20_000
SVM_C_VALUES = [0.01, 0.1, 1.0]
TOP_FRACTION = 0.01
SVM_MAX_ITERATIONS = 10_000

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "NK_cell_dataset").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "NK_cell_dataset" / "NK_cell_dataset"
FCS_DIR = DATA_ROOT / "NK_cell_dataset" / GATE
LABELS_PATH = DATA_ROOT / "NK_fcs_samples_with_labels.csv"
MARKERS_PATH = DATA_ROOT / "NK_markers.csv"
SPLITS_PATH = PROJECT_ROOT / "results" / "tables" / "task4_donor_splits.csv"
SVM_PREDICTIONS_PATH = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / f"task4_svm_predictions_{GATE}_{RUN_MODE}.csv"
)
SVM_SELECTION_PATH = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / f"task4_svm_selection_{GATE}_{RUN_MODE}.csv"
)

for required_path in (FCS_DIR, LABELS_PATH, MARKERS_PATH, SPLITS_PATH):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Erwarteter Pfad fehlt: {required_path}. "
            "Gegebenenfalls zuerst 04a_data_qc_and_splits.ipynb ausführen."
        )

print(
    f"Modus: {RUN_MODE}; Gate: {GATE}; Analysesplits: {len(ANALYSIS_SPLIT_IDS)}; "
    f"Training: {'aktiv' if RUN_TRAINING else 'deaktiviert'}"
)

Modus: smoke; Gate: gated_NK; Analysesplits: 2; Training: deaktiviert


## 2. Marker, Labels und gemeinsame Splits laden

In [2]:
with MARKERS_PATH.open(newline="", encoding="utf-8-sig") as stream:
    markers = next(csv.reader(stream))

label_table = pd.read_csv(LABELS_PATH)
label_table["donor_id"] = label_table["fcs_filename"].str.replace(
    r"_NK\.fcs$", "", regex=True
)
label_table["label"] = label_table["label"].astype(int)

fcs_paths = sorted(FCS_DIR.glob("*.fcs"))
fcs_table = pd.DataFrame(
    {
        "donor_id": [path.stem.removesuffix(GATE_SUFFIX) for path in fcs_paths],
        "fcs_path": fcs_paths,
    }
)
sample_table = (
    label_table[["donor_id", "label"]]
    .merge(fcs_table, on="donor_id", validate="one_to_one")
    .sort_values("donor_id")
    .reset_index(drop=True)
)
donor_splits = pd.read_csv(SPLITS_PATH)

assert len(markers) == 37
assert len(fcs_paths) == 20
assert len(sample_table) == 20
assert set(sample_table["donor_id"]) == set(donor_splits["donor_id"])
split_labels = donor_splits[["donor_id", "label"]].drop_duplicates()
expected_labels = sample_table[["donor_id", "label"]]
assert split_labels.merge(
    expected_labels, on=["donor_id", "label"], validate="one_to_one"
).shape[0] == len(expected_labels)
assert set(TECHNICAL_SPLIT_IDS).issubset(set(donor_splits["split_id"]))
assert donor_splits.groupby(["split_id", "donor_id"]).size().eq(1).all()

display(sample_table[["donor_id", "label"]])

,donor_id,label
0,a_001,1
1,a_002,1
2,a_003,0
3,a_004,0
4,a_005,1
5,a_006,0
6,a_007,1
7,a_009,0
8,a_010,0
9,a_011,0


## 3. Transformierte Daten spenderweise laden

Die Daten bleiben als getrennte Arrays pro Spender erhalten. Dadurch kann ein Spender nur vollständig einer Trainings-, Validierungs- oder Testmenge zugeordnet werden.

In [3]:
def load_transformed_fcs(path: Path, selected_markers: list[str]) -> np.ndarray:
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            message=r"FCS file .* reported incorrect data offset.*",
        )
        sample = fk.Sample(str(path), ignore_offset_error=True)

    frame = sample.as_dataframe(source="raw")
    short_names = pd.Index(sample.pns_labels, name="marker")
    if not short_names.is_unique:
        raise ValueError(f"Doppelte FCS-Kurznamen in {path.name}.")
    frame.columns = short_names
    missing_markers = [marker for marker in selected_markers if marker not in frame.columns]
    if missing_markers:
        raise ValueError(f"Fehlende Marker in {path.name}: {missing_markers}")

    raw_values = frame.loc[:, selected_markers].to_numpy(dtype=np.float32, copy=True)
    transformed_values = np.arcsinh(raw_values / COFACTOR)
    if not np.isfinite(transformed_values).all():
        raise ValueError(f"Nicht-endliche Werte in {path.name}.")
    return transformed_values


data_by_donor = {
    row.donor_id: load_transformed_fcs(row.fcs_path, markers)
    for row in sample_table.itertuples(index=False)
}
label_by_donor = sample_table.set_index("donor_id")["label"].to_dict()

loaded_overview = pd.DataFrame(
    [
        {
            "donor_id": donor_id,
            "label": label_by_donor[donor_id],
            "cells": len(values),
            "markers": values.shape[1],
        }
        for donor_id, values in data_by_donor.items()
    ]
)
assert loaded_overview["cells"].sum() == EXPECTED_EVENT_COUNTS[GATE]
assert loaded_overview["markers"].eq(37).all()
display(loaded_overview)

,donor_id,label,cells,markers
0,a_001,1,6320,37
1,a_002,1,15809,37
2,a_003,0,11438,37
3,a_004,0,17443,37
4,a_005,1,25226,37
5,a_006,0,5652,37
6,a_007,1,9819,37
7,a_009,0,20401,37
8,a_010,0,9092,37
9,a_011,0,31424,37


## 4. Spenderbalanciertes Sampling und fold-sichere Skalierung

Für den technischen Test werden nur aus den Trainingsspendern gleich viele Zellen ohne Zurücklegen gezogen. Der `StandardScaler` sieht ausschließlich diese Zellen der inneren Trainingsspender. Innere Validierungs- und äußere Testspender werden vollständig und mit unveränderten Trainingsparametern transformiert.

In [4]:
def stack_sampled_donors(
    donor_ids: list[str],
    cells_per_donor: int,
    rng: np.random.Generator,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    sampled_values = []
    sampled_labels = []
    sampled_donor_ids = []

    for donor_id in sorted(donor_ids):
        donor_values = data_by_donor[donor_id]
        if len(donor_values) < cells_per_donor:
            raise ValueError(
                f"{donor_id} besitzt nur {len(donor_values)} Zellen; "
                f"angefordert sind {cells_per_donor}."
            )
        indices = rng.choice(len(donor_values), size=cells_per_donor, replace=False)
        sampled_values.append(donor_values[indices])
        sampled_labels.append(
            np.full(cells_per_donor, label_by_donor[donor_id], dtype=np.int8)
        )
        sampled_donor_ids.append(
            np.full(cells_per_donor, donor_id, dtype=object)
        )

    return (
        np.concatenate(sampled_values),
        np.concatenate(sampled_labels),
        np.concatenate(sampled_donor_ids),
    )


def stack_all_donors(
    donor_ids: list[str],
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    values = [data_by_donor[donor_id] for donor_id in sorted(donor_ids)]
    labels = [
        np.full(len(data_by_donor[donor_id]), label_by_donor[donor_id], dtype=np.int8)
        for donor_id in sorted(donor_ids)
    ]
    cell_donor_ids = [
        np.full(len(data_by_donor[donor_id]), donor_id, dtype=object)
        for donor_id in sorted(donor_ids)
    ]
    return np.concatenate(values), np.concatenate(labels), np.concatenate(cell_donor_ids)


def prepare_inner_fold(
    split_id: int,
    validation_fold: int,
    cells_per_donor: int,
) -> dict[str, object]:
    split = donor_splits.loc[donor_splits["split_id"] == split_id].copy()
    inner_train_ids = split.loc[
        (split["outer_partition"] == "train")
        & (split["inner_fold"] != validation_fold),
        "donor_id",
    ].tolist()
    inner_validation_ids = split.loc[
        (split["outer_partition"] == "train")
        & (split["inner_fold"] == validation_fold),
        "donor_id",
    ].tolist()
    outer_test_ids = split.loc[
        split["outer_partition"] == "test", "donor_id"
    ].tolist()

    donor_sets = [set(inner_train_ids), set(inner_validation_ids), set(outer_test_ids)]
    assert not donor_sets[0] & donor_sets[1]
    assert not donor_sets[0] & donor_sets[2]
    assert not donor_sets[1] & donor_sets[2]
    assert set.union(*donor_sets) == set(data_by_donor)

    split_seed = int(split["split_seed"].iloc[0])
    rng = np.random.default_rng(
        split_seed + SAMPLING_SEED_OFFSET + validation_fold
    )
    train = stack_sampled_donors(inner_train_ids, cells_per_donor, rng)
    validation = stack_all_donors(inner_validation_ids)

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train[0])
    validation_scaled = scaler.transform(validation[0])

    assert np.isfinite(train_scaled).all()
    assert np.isfinite(validation_scaled).all()
    assert np.max(np.abs(train_scaled.mean(axis=0))) < 1e-5

    return {
        "split_id": split_id,
        "validation_fold": validation_fold,
        "train": (train_scaled, train[1], train[2]),
        "validation": (validation_scaled, validation[1], validation[2]),
        "outer_test_ids": outer_test_ids,
        "scaler": scaler,
    }


## 5. Technischer Test auf zwei Splits

In [5]:
technical_fold_data = [
    prepare_inner_fold(
        split_id=split_id,
        validation_fold=TECHNICAL_INNER_FOLD,
        cells_per_donor=TECHNICAL_CELLS_PER_DONOR,
    )
    for split_id in TECHNICAL_SPLIT_IDS
]

technical_records = []
for fold_data in technical_fold_data:
    test_unscaled = stack_all_donors(fold_data["outer_test_ids"])
    test = (
        fold_data["scaler"].transform(test_unscaled[0]),
        test_unscaled[1],
        test_unscaled[2],
    )
    partitions = {
        "train": fold_data["train"],
        "validation": fold_data["validation"],
        "test": test,
    }
    for partition, (values, labels, donor_ids) in partitions.items():
        technical_records.append(
            {
                "split_id": fold_data["split_id"],
                "partition": partition,
                "donors": len(np.unique(donor_ids)),
                "cells": len(values),
                "CMV− cells": int((labels == 0).sum()),
                "CMV+ cells": int((labels == 1).sum()),
                "finite": bool(np.isfinite(values).all()),
            }
        )

technical_summary = pd.DataFrame(technical_records)
assert technical_summary["finite"].all()
assert technical_summary.loc[technical_summary["partition"] == "test", "donors"].eq(6).all()
display(technical_summary)

,split_id,partition,donors,cells,CMV− cells,CMV+ cells,finite
0,0,train,9,18000,10000,8000,True
1,0,validation,5,80775,42862,37913,True
2,0,test,6,69612,38870,30742,True
3,1,train,9,18000,10000,8000,True
4,1,validation,5,72347,43029,29318,True
5,1,test,6,75833,47920,27913,True


## Abschluss von Schritt 2

Der technische Datenfluss funktioniert auf zwei festen `gated_NK`-Splits. Sampling, Transformation, innere Validierung, äußerer Test und trainingsfold-sichere Standardisierung sind damit vorbereitet.

## 6. Lineare Single-Cell-SVM

Jede Trainingszelle erhält das Label ihres Spenders. Für jedes `C` wird die Leistung in den drei inneren Validierungsfolds ausschließlich auf Spenderebene bewertet. Die kontinuierlichen Zell-Scores werden pro Spender als Mittelwert der höchsten 1 % aggregiert.

Nach Auswahl von `C` wird aus den zusammengeführten inneren Out-of-fold-Spendervorhersagen ein Schwellenwert nach dem Youden-Kriterium bestimmt. Anschließend wird die SVM auf allen 14 äußeren Trainingsspendern neu gefittet und genau einmal auf den sechs äußeren Testspendern ausgewertet.

In [6]:
def aggregate_top_fraction(
    cell_scores: np.ndarray,
    cell_donor_ids: np.ndarray,
    top_fraction: float = TOP_FRACTION,
) -> pd.DataFrame:
    if not 0 < top_fraction <= 1:
        raise ValueError("top_fraction muss im Intervall (0, 1] liegen.")

    records = []
    for donor_id in sorted(np.unique(cell_donor_ids)):
        donor_scores = cell_scores[cell_donor_ids == donor_id]
        top_count = max(1, int(np.ceil(top_fraction * len(donor_scores))))
        top_scores = np.partition(donor_scores, -top_count)[-top_count:]
        records.append(
            {
                "donor_id": donor_id,
                "y_true": label_by_donor[donor_id],
                "score": float(top_scores.mean()),
                "top_cell_count": top_count,
            }
        )
    return pd.DataFrame(records)


def select_youden_threshold(y_true: np.ndarray, scores: np.ndarray) -> float:
    false_positive_rate, true_positive_rate, thresholds = roc_curve(y_true, scores)
    youden_index = true_positive_rate - false_positive_rate
    finite = np.isfinite(thresholds)
    if not finite.any():
        raise ValueError("Kein endlicher Schwellenwert verfügbar.")
    finite_indices = np.flatnonzero(finite)
    best_index = finite_indices[int(np.argmax(youden_index[finite]))]
    return float(thresholds[best_index])


def trapezoidal_pr_auc(y_true: np.ndarray, scores: np.ndarray) -> float:
    precision, recall, _ = precision_recall_curve(y_true, scores)
    return float(auc(recall, precision))


def select_svm_hyperparameters(
    split_id: int,
    cells_per_donor: int,
) -> tuple[float, float, pd.DataFrame]:
    candidate_records = []
    out_of_fold_by_c = {}

    for c_value in SVM_C_VALUES:
        fold_predictions = []
        for validation_fold in range(3):
            fold_data = prepare_inner_fold(
                split_id=split_id,
                validation_fold=validation_fold,
                cells_per_donor=cells_per_donor,
            )
            train_values, train_labels, _ = fold_data["train"]
            validation_values, _, validation_donors = fold_data["validation"]
            split_seed = int(
                donor_splits.loc[donor_splits["split_id"] == split_id, "split_seed"].iloc[0]
            )
            classifier = LinearSVC(
                C=c_value,
                dual="auto",
                max_iter=SVM_MAX_ITERATIONS,
                random_state=split_seed + validation_fold,
            )
            classifier.fit(train_values, train_labels)
            validation_scores = classifier.decision_function(validation_values)
            donor_predictions = aggregate_top_fraction(
                validation_scores, validation_donors
            )
            donor_predictions["inner_fold"] = validation_fold
            fold_predictions.append(donor_predictions)
            candidate_records.append(
                {
                    "split_id": split_id,
                    "C": c_value,
                    "inner_fold": validation_fold,
                    "validation_roc_auc": roc_auc_score(
                        donor_predictions["y_true"], donor_predictions["score"]
                    ),
                }
            )

        out_of_fold_by_c[c_value] = pd.concat(fold_predictions, ignore_index=True)

    candidate_results = pd.DataFrame(candidate_records)
    mean_auc_by_c = (
        candidate_results.groupby("C")["validation_roc_auc"].mean().sort_index()
    )
    best_c = float(
        mean_auc_by_c.reset_index()
        .sort_values(["validation_roc_auc", "C"], ascending=[False, True])
        .iloc[0]["C"]
    )
    best_out_of_fold = out_of_fold_by_c[best_c]
    threshold = select_youden_threshold(
        best_out_of_fold["y_true"].to_numpy(),
        best_out_of_fold["score"].to_numpy(),
    )
    return best_c, threshold, candidate_results


def fit_and_predict_outer_split(
    split_id: int,
    cells_per_donor: int,
    best_c: float,
    threshold: float,
) -> pd.DataFrame:
    split = donor_splits.loc[donor_splits["split_id"] == split_id]
    train_ids = split.loc[split["outer_partition"] == "train", "donor_id"].tolist()
    test_ids = split.loc[split["outer_partition"] == "test", "donor_id"].tolist()
    assert not set(train_ids) & set(test_ids)

    split_seed = int(split["split_seed"].iloc[0])
    rng = np.random.default_rng(split_seed + SAMPLING_SEED_OFFSET + 100)
    train_values, train_labels, _ = stack_sampled_donors(
        train_ids, cells_per_donor, rng
    )
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_values)
    classifier = LinearSVC(
        C=best_c,
        dual="auto",
        max_iter=SVM_MAX_ITERATIONS,
        random_state=split_seed,
    )
    classifier.fit(train_scaled, train_labels)

    test_records = []
    for donor_id in sorted(test_ids):
        donor_values = scaler.transform(data_by_donor[donor_id])
        donor_cell_scores = classifier.decision_function(donor_values)
        top_count = max(1, int(np.ceil(TOP_FRACTION * len(donor_cell_scores))))
        top_scores = np.partition(donor_cell_scores, -top_count)[-top_count:]
        donor_score = float(top_scores.mean())
        test_records.append(
            {
                "method": "linear_single_cell_svm",
                "gate": GATE,
                "run_mode": RUN_MODE,
                "split_id": split_id,
                "split_seed": split_seed,
                "donor_id": donor_id,
                "y_true": label_by_donor[donor_id],
                "score": donor_score,
                "decision_threshold": threshold,
                "y_pred": int(donor_score >= threshold),
                "best_C": best_c,
                "top_fraction": TOP_FRACTION,
                "top_cell_count": top_count,
                "train_cells_per_donor": cells_per_donor,
            }
        )
    return pd.DataFrame(test_records)

## 7. Deterministischer SVM-Smoke-Test

Im voreingestellten Smoke-Modus werden zwei Splits und 2.000 Trainingszellen je Spender verwendet. Im Full-Modus laufen die 100 gemeinsamen Splits auf `gated_alive` mit 10.000 Trainingszellen je Spender. Ergebnisse werden nach jedem Split gespeichert und können fortgesetzt werden.

In [7]:
svm_predictions_by_split = []
svm_selection_by_split = []
completed_split_ids = set()
if RUN_TRAINING and SVM_PREDICTIONS_PATH.exists() and SVM_SELECTION_PATH.exists():
    previous_predictions = pd.read_csv(SVM_PREDICTIONS_PATH)
    previous_selection = pd.read_csv(SVM_SELECTION_PATH)
    if (
        {"gate", "run_mode"}.issubset(previous_predictions.columns)
        and previous_predictions["gate"].eq(GATE).all()
        and previous_predictions["run_mode"].eq(RUN_MODE).all()
    ):
        previous_predictions = previous_predictions.loc[
            ~previous_predictions["split_id"].isin(FORCE_SPLIT_IDS)
        ]
        previous_selection = previous_selection.loc[
            ~previous_selection["split_id"].isin(FORCE_SPLIT_IDS)
        ]
        svm_predictions_by_split.append(previous_predictions)
        svm_selection_by_split.append(previous_selection)
        completed_split_ids = set(previous_predictions["split_id"])
        print(f"Vorhandene SVM-Ergebnisse geladen: {len(completed_split_ids)} Splits.")

if RUN_TRAINING:
  for split_id in ANALYSIS_SPLIT_IDS:
    if split_id in completed_split_ids:
        continue
    best_c, threshold, candidate_results = select_svm_hyperparameters(
        split_id=split_id,
        cells_per_donor=TRAIN_CELLS_PER_DONOR,
    )
    split_predictions = fit_and_predict_outer_split(
        split_id=split_id,
        cells_per_donor=TRAIN_CELLS_PER_DONOR,
        best_c=best_c,
        threshold=threshold,
    )
    svm_predictions_by_split.append(split_predictions)
    candidate_results["gate"] = GATE
    candidate_results["run_mode"] = RUN_MODE
    candidate_results["selected"] = candidate_results["C"].eq(best_c)
    svm_selection_by_split.append(candidate_results)
    svm_predictions = pd.concat(svm_predictions_by_split, ignore_index=True)
    svm_selection = pd.concat(svm_selection_by_split, ignore_index=True)
    svm_predictions.to_csv(SVM_PREDICTIONS_PATH, index=False)
    svm_selection.to_csv(SVM_SELECTION_PATH, index=False)
    print(f"Split {split_id} abgeschlossen.")
else:
    print("Training deaktiviert; vorhandene SVM-Ergebnisse werden geladen.")
    svm_predictions_by_split = [pd.read_csv(SVM_PREDICTIONS_PATH)]
    svm_selection_by_split = [pd.read_csv(SVM_SELECTION_PATH)]

svm_predictions = pd.concat(svm_predictions_by_split, ignore_index=True)
svm_selection = pd.concat(svm_selection_by_split, ignore_index=True)

assert len(svm_predictions) == len(ANALYSIS_SPLIT_IDS) * 6
assert set(svm_predictions["split_id"]) == set(ANALYSIS_SPLIT_IDS)
assert svm_predictions.groupby(["split_id", "donor_id"]).size().eq(1).all()
assert svm_predictions["gate"].eq(GATE).all()
assert svm_predictions["run_mode"].eq(RUN_MODE).all()
assert svm_predictions["train_cells_per_donor"].eq(TRAIN_CELLS_PER_DONOR).all()
assert np.isfinite(svm_predictions["score"]).all()
assert np.isfinite(svm_predictions["decision_threshold"]).all()
expected_test = donor_splits.loc[
    donor_splits["split_id"].isin(ANALYSIS_SPLIT_IDS)
    & donor_splits["outer_partition"].eq("test"),
    ["split_id", "donor_id", "label"],
].rename(columns={"label": "y_true"})
assert svm_predictions[["split_id", "donor_id", "y_true"]].merge(
    expected_test, on=["split_id", "donor_id", "y_true"], validate="one_to_one"
).shape[0] == len(expected_test) == len(svm_predictions)

svm_metrics = (
    svm_predictions.groupby("split_id")
    .apply(
        lambda group: pd.Series(
            {
                "roc_auc": roc_auc_score(group["y_true"], group["score"]),
                "pr_auc": trapezoidal_pr_auc(
                    group["y_true"].to_numpy(), group["score"].to_numpy()
                ),
                "balanced_accuracy": balanced_accuracy_score(
                    group["y_true"], group["y_pred"]
                ),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

display(svm_selection)
display(svm_predictions)
display(svm_metrics)
print(f"Vorhersagen gespeichert: {SVM_PREDICTIONS_PATH.relative_to(PROJECT_ROOT)}")

Training deaktiviert; vorhandene SVM-Ergebnisse werden geladen.


,split_id,C,inner_fold,validation_roc_auc,gate,run_mode,selected
0,0,0.01,0,0.666667,gated_NK,smoke,True
1,0,0.01,1,1.000000,gated_NK,smoke,True
2,0,0.01,2,1.000000,gated_NK,smoke,True
3,0,0.10,0,0.666667,gated_NK,smoke,False
4,0,0.10,1,1.000000,gated_NK,smoke,False
5,0,0.10,2,1.000000,gated_NK,smoke,False
6,0,1.00,0,0.666667,gated_NK,smoke,False
7,0,1.00,1,1.000000,gated_NK,smoke,False
8,0,1.00,2,1.000000,gated_NK,smoke,False
9,1,0.01,0,0.666667,gated_NK,smoke,True


,method,gate,run_mode,split_id,split_seed,donor_id,y_true,score,decision_threshold,y_pred,best_C,top_fraction,top_cell_count,train_cells_per_donor
0,linear_single_cell_svm,gated_NK,smoke,0,3003105692,a_002,1,1.095853,0.995085,1,0.01,0.01,159,2000
1,linear_single_cell_svm,gated_NK,smoke,0,3003105692,a_004,0,0.733303,0.995085,0,0.01,0.01,175,2000
2,linear_single_cell_svm,gated_NK,smoke,0,3003105692,a_006,0,0.953807,0.995085,0,0.01,0.01,57,2000
3,linear_single_cell_svm,gated_NK,smoke,0,3003105692,a_1a,0,1.175358,0.995085,1,0.01,0.01,71,2000
4,linear_single_cell_svm,gated_NK,smoke,0,3003105692,a_2a,0,1.104615,0.995085,1,0.01,0.01,88,2000
5,linear_single_cell_svm,gated_NK,smoke,0,3003105692,a_4a,1,1.541579,0.995085,1,0.01,0.01,150,2000
6,linear_single_cell_svm,gated_NK,smoke,1,976400780,a_002,1,1.033621,1.406807,0,0.01,0.01,159,2000
7,linear_single_cell_svm,gated_NK,smoke,1,976400780,a_003,0,1.244284,1.406807,0,0.01,0.01,115,2000
8,linear_single_cell_svm,gated_NK,smoke,1,976400780,a_004,0,0.811553,1.406807,0,0.01,0.01,175,2000
9,linear_single_cell_svm,gated_NK,smoke,1,976400780,a_006,0,1.155191,1.406807,0,0.01,0.01,57,2000


,split_id,roc_auc,pr_auc,balanced_accuracy
0,0,0.75,0.708333,0.75
1,1,0.75,0.708333,0.75


Vorhersagen gespeichert: results/tables/task4_svm_predictions_gated_NK_smoke.csv


## Abschluss von Schritt 3

Die lineare Single-Cell-SVM ist vollständig in die spenderweise Auswertungspipeline integriert und auf zwei kleinen `gated_NK`-Splits technisch geprüft. Die gespeicherten Smoke-Test-Ergebnisse sind ausdrücklich nicht für den finalen Methodenvergleich bestimmt. Der vollständige Lauf auf `gated_alive` folgt erst nach Implementierung und Einzelprüfung aller drei Hauptmethoden.